In [1]:
tabla='ctpco10'
col_tabla='proconfec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:

fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=6", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=6"

c1= text(query)
connection2.execute(c1)

2023-05-24


/tmp/ipykernel_101834/4108644430.py:10: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  connection2.execute(c1)


In [5]:
fecha='2023-05-25'

In [6]:

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)




connection0.close()



In [7]:
base2.shape

(704471, 36)

In [8]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704471 entries, 0 to 704470
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   proconoricenasicod       704471 non-null  object        
 1   proconcenasicod          704471 non-null  object        
 2   proconarehoscod          704471 non-null  object        
 3   proconservhoscod         704471 non-null  object        
 4   proconactcod             704471 non-null  object        
 5   proconactespcod          704471 non-null  object        
 6   procontipdocidenpercod   704471 non-null  object        
 7   proconperasisdocidennum  704471 non-null  object        
 8   proconfec                704471 non-null  datetime64[ns]
 9   proconturhorini          704471 non-null  object        
 10  proconturhorfin          704471 non-null  object        
 11  concod                   704471 non-null  object        
 12  proconcancupcitv

In [9]:
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

471

In [10]:


borrando=f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'"
borrado = connection3.execute(borrando)


query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


In [11]:


query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN proconoricenasicod TYPE character(1),
ALTER COLUMN proconcenasicod TYPE character(3),
ALTER COLUMN proconarehoscod TYPE character(2),
ALTER COLUMN proconservhoscod TYPE character(3),
ALTER COLUMN proconactcod TYPE character(2),
ALTER COLUMN proconactespcod TYPE character(3),
ALTER COLUMN procontipdocidenpercod TYPE character(1),
ALTER COLUMN proconperasisdocidennum TYPE character(10),
ALTER COLUMN proconfec TYPE date,
ALTER COLUMN proconturhorini TYPE timestamp,
ALTER COLUMN proconturhorfin TYPE timestamp,
ALTER COLUMN concod TYPE character(5),
ALTER COLUMN proconcancupcitvol TYPE numeric(3,0),
ALTER COLUMN proconcancupreci TYPE numeric(3,0),
ALTER COLUMN proconcancupinte TYPE numeric(3,0),
ALTER COLUMN proconcancupcitdia TYPE numeric(3,0),
ALTER COLUMN proconcancuptopnuev TYPE numeric(3,0),
ALTER COLUMN proconcancuptopadic TYPE numeric(3,0),
ALTER COLUMN proconcancuptoprefe TYPE numeric(3,0),
ALTER COLUMN proconcancuptopterc TYPE numeric(3,0),
ALTER COLUMN proconcanotorcitvol TYPE numeric(3,0),
ALTER COLUMN proconcanotorreci TYPE numeric(3,0),
ALTER COLUMN proconcanotorinte TYPE numeric(3,0),
ALTER COLUMN proconcanotortopnuev TYPE numeric(3,0),
ALTER COLUMN proconcanotortopadic TYPE numeric(3,0),
ALTER COLUMN proconcanotortoprefe TYPE numeric(3,0),
ALTER COLUMN proconcanotortopterc TYPE numeric(3,0),
ALTER COLUMN proconcanciteli TYPE numeric(3,0),
ALTER COLUMN proconcancitate TYPE numeric(3,0),
ALTER COLUMN proconcancitrep TYPE numeric(3,0),
ALTER COLUMN proconcandemins TYPE numeric(3,0),
ALTER COLUMN proconestcom TYPE character(1),
ALTER COLUMN proconestregcod TYPE character(1),
ALTER COLUMN proconcancupcitref TYPE numeric(3,0),
ALTER COLUMN proconcanotorcitref TYPE numeric(3,0),
ALTER COLUMN proconcanpachor TYPE numeric(2,0);




INSERT INTO {tabla} 
(proconoricenasicod,proconcenasicod,proconarehoscod,proconservhoscod,proconactcod,proconactespcod,procontipdocidenpercod,proconperasisdocidennum,proconfec,proconturhorini,proconturhorfin,concod,proconcancupcitvol,proconcancupreci,proconcancupinte,proconcancupcitdia,proconcancuptopnuev,proconcancuptopadic,proconcancuptoprefe,proconcancuptopterc,proconcanotorcitvol,proconcanotorreci,proconcanotorinte,proconcanotortopnuev,proconcanotortopadic,proconcanotortoprefe,proconcanotortopterc,proconcanciteli,proconcancitate,proconcancitrep,proconcandemins,proconestcom,proconestregcod,proconcancupcitref,proconcanotorcitref,proconcanpachor) 

SELECT 
proconoricenasicod,proconcenasicod,proconarehoscod,proconservhoscod,proconactcod,proconactespcod,procontipdocidenpercod,proconperasisdocidennum,proconfec,proconturhorini,proconturhorfin,concod,proconcancupcitvol,proconcancupreci,proconcancupinte,proconcancupcitdia,proconcancuptopnuev,proconcancuptopadic,proconcancuptoprefe,proconcancuptopterc,proconcanotorcitvol,proconcanotorreci,proconcanotorinte,proconcanotortopnuev,proconcanotortopadic,proconcanotortoprefe,proconcanotortopterc,proconcanciteli,proconcancitate,proconcancitrep,proconcandemins,proconestcom,proconestregcod,proconcancupcitref,proconcanotorcitref,proconcanpachor



FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)


In [12]:

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)


#BORRAMOS LAS TABLAS
query2=f"""
SELECT COUNT(*) FROM {tabla} ;
"""
c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection3.close() 


In [ ]:
connection1.close() 
connection2.close() 
connection3.close() 
